In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Dataset 

In [16]:
file_path = "Merged_TSQIC_REDCap.xlsx" 
merged_final_tsqic_redcap_access = pd.read_excel(file_path)
merged_final_tsqic_redcap_access

,id,operation_date,Complication,Grade,GradeLetter,readmission_30d,DischargeDate,los,redcap_event_name,redcap_repeat_instrument,redcap_repeat_instance,path_proximalmargin,path_distalmargin,path_circummargin,path_lymph_total,intraop_complications
0,1,NaT,NaN,NaN,NaN,NaN,NaT,NaN,surgery_arm_1,surgery_esd_emr,1.0,NaN,NaN,NaN,NaN,0.0
1,1,NaT,NaN,NaN,NaN,NaN,NaT,NaN,surgery_arm_1,surgical_pathology,1.0,1.0,1.0,1.0,34.0,NaN
2,2,NaT,NaN,NaN,NaN,NaN,NaT,NaN,surgery_arm_1,surgical_pathology,1.0,1.0,1.0,1.0,41.0,NaN
3,2,NaT,NaN,NaN,NaN,NaN,NaT,NaN,surgery_arm_1,surgery_esd_emr,1.0,NaN,NaN,NaN,NaN,0.0
4,2,NaT,NaN,NaN,NaN,NaN,NaT,NaN,1_month_postop_arm_1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5408,1764,NaT,NaN,NaN,NaN,NaN,NaT,NaN,surgery_arm_1,surgical_pathology,1.0,1.0,1.0,2.0,46.0,NaN
5409,1766,NaT,NaN,NaN,NaN,NaN,NaT,NaN,surgery_arm_1,surgery_esd_emr,1.0,NaN,NaN,NaN,NaN,0.0
5410,1766,NaT,NaN,NaN,NaN,NaN,NaT,NaN,surgery_arm_1,surgical_pathology,2.0,1.0,1.0,1.0,50.0,NaN
5411,1770,NaT,NaN,NaN,NaN,NaN,NaT,NaN,surgery_arm_1,surgery_esd_emr,1.0,NaN,NaN,NaN,NaN,0.0


In [17]:
len(merged_final_tsqic_redcap_access['id'].unique())

1270

# Analysis

In [18]:
# Group data by patient ID
patient_groups = merged_final_tsqic_redcap_access.groupby('id')

# Track results for each patient
patient_outcomes = {}
patient_missing_data = {}
patient_failed_criteria = {}
failed_criteria_counts = {
    'path_proximalmargin': 0,
    'path_distalmargin': 0,
    'path_circummargin': 0,
    'path_lymph_total': 0,
    'intraop_complications': 0,
    'Grade': 0,
    'Complication_not_leak': 0,
    'readmission_30d': 0,
    'los': 0
}

In [19]:
# Process each patient
for patient_id, patient_data in patient_groups:
    # First, check if any criterion is completely missing across all rows
    has_data_for_criterion = {
        'path_proximalmargin': False,
        'path_distalmargin': False,
        'path_circummargin': False,
        'path_lymph_total': False,
        'intraop_complications': False,
        'Grade': False,
        'Complication_not_leak': False,
        'readmission_30d': False,
        'los': False
    }
    
    # Track if any criterion is violated
    criteria_violated = {
        'path_proximalmargin': False,
        'path_distalmargin': False,
        'path_circummargin': False,
        'path_lymph_total': False,
        'intraop_complications': False,
        'Grade': False,
        'Complication_not_leak': False,
        'readmission_30d': False,
        'los': False
    }
    
    # First pass: check if each criterion has any data at all across all rows
    for _, row in patient_data.iterrows():
        if pd.notna(row['path_proximalmargin']):
            has_data_for_criterion['path_proximalmargin'] = True
            
        if pd.notna(row['path_distalmargin']):
            has_data_for_criterion['path_distalmargin'] = True
            
        if pd.notna(row['path_circummargin']):
            has_data_for_criterion['path_circummargin'] = True
            
        if pd.notna(row['path_lymph_total']):
            has_data_for_criterion['path_lymph_total'] = True
            
        if pd.notna(row['intraop_complications']):
            has_data_for_criterion['intraop_complications'] = True
            
        if pd.notna(row['Grade']):
            has_data_for_criterion['Grade'] = True
            
        if pd.notna(row['Complication']):
            has_data_for_criterion['Complication_not_leak'] = True
            
        if pd.notna(row['readmission_30d']):
            has_data_for_criterion['readmission_30d'] = True
            
        if pd.notna(row['los']):
            has_data_for_criterion['los'] = True
    
    # Check if any criterion is completely missing across all rows
    missing_criteria = [criterion for criterion, has_data in has_data_for_criterion.items() if not has_data]
    has_missing_data = len(missing_criteria) > 0
    
    if has_missing_data:
        # Record which data is missing and exclude from outcome analysis
        patient_missing_data[patient_id] = missing_criteria
        continue
    
    # Second pass: now that we know all criteria have data somewhere, check if any criteria are violated
    criteria_met = {
        'path_proximalmargin': False,
        'path_distalmargin': False,
        'path_circummargin': False,
        'path_lymph_total': False,
        'intraop_complications': False,
        'Grade': False,
        'Complication_not_leak': False,
        'readmission_30d': False,
        'los': False
    }
    
    for _, row in patient_data.iterrows():
        # Check each criterion for violations
        if pd.notna(row['path_proximalmargin']):
            if row['path_proximalmargin'] == 1:
                criteria_met['path_proximalmargin'] = True
            else:
                criteria_violated['path_proximalmargin'] = True
                
        if pd.notna(row['path_distalmargin']):
            if row['path_distalmargin'] == 1:
                criteria_met['path_distalmargin'] = True
            else:
                criteria_violated['path_distalmargin'] = True
                
        if pd.notna(row['path_circummargin']):
            if row['path_circummargin'] == 1:
                criteria_met['path_circummargin'] = True
            else:
                criteria_violated['path_circummargin'] = True
                
        if pd.notna(row['path_lymph_total']):
            if row['path_lymph_total'] >= 20:
                criteria_met['path_lymph_total'] = True
            else:
                criteria_violated['path_lymph_total'] = True
                
        if pd.notna(row['intraop_complications']):
            if row['intraop_complications'] == 0:
                criteria_met['intraop_complications'] = True
            else:
                criteria_violated['intraop_complications'] = True
                
        if pd.notna(row['Grade']):
            if row['Grade'] < 3:
                criteria_met['Grade'] = True
            else:
                criteria_violated['Grade'] = True
                
        if pd.notna(row['Complication']):
            if row['Complication'] != 'Leak':
                criteria_met['Complication_not_leak'] = True
            else:
                criteria_violated['Complication_not_leak'] = True
                
        if pd.notna(row['readmission_30d']):
            if row['readmission_30d'] == 0:
                criteria_met['readmission_30d'] = True
            else:
                criteria_violated['readmission_30d'] = True
                
        if pd.notna(row['los']):
            if row['los'] < 14:
                criteria_met['los'] = True
            else:
                criteria_violated['los'] = True
    
    # For textbook outcome:
    # 1. Patient must have at least one value meeting each criterion
    all_criteria_met = all(criteria_met.values())
    # 2. Patient must not have any violation of any criterion
    no_criteria_violated = not any(criteria_violated.values())
    
    textbook_outcome = all_criteria_met and no_criteria_violated
    patient_outcomes[patient_id] = textbook_outcome
    
    # If patient doesn't achieve textbook outcome, record which criteria they failed
    if not textbook_outcome:
        patient_failed_criteria[patient_id] = {}
        
        # Record criteria that were not met (no positive value found)
        for criterion, met in criteria_met.items():
            if not met:
                failed_criteria_counts[criterion] += 1
                patient_failed_criteria[patient_id][criterion] = "no_positive_value"
        
        # Record criteria that were violated
        for criterion, violated in criteria_violated.items():
            if violated:
                failed_criteria_counts[criterion] += 1
                patient_failed_criteria[patient_id][criterion] = "criterion_violated"

In [20]:
patient_outcomes

{79: False,
 89: False,
 102: True,
 127: False,
 128: False,
 141: False,
 190: False,
 202: True,
 228: False,
 230: False,
 261: False,
 269: False,
 279: False,
 459: True,
 473: False,
 497: False,
 508: False,
 517: False,
 551: False,
 560: False,
 563: False,
 566: False,
 567: False,
 579: False,
 587: False,
 589: False,
 590: False,
 592: False,
 597: True,
 603: False,
 614: False,
 615: False,
 648: False,
 651: False,
 653: False,
 659: True,
 661: False,
 663: False,
 665: False,
 666: False,
 667: False,
 668: False,
 669: False,
 670: False,
 671: True,
 672: False,
 675: False,
 677: False,
 679: False,
 682: False,
 683: False,
 684: True,
 685: True,
 688: False,
 689: False,
 690: False,
 694: True,
 699: False,
 706: True,
 707: True,
 714: True,
 732: False,
 734: False,
 735: False,
 736: False,
 738: False,
 742: False,
 744: False,
 747: False,
 750: False,
 751: False,
 753: False,
 754: False,
 755: False,
 757: True,
 758: False,
 760: False,
 764: False,
 

In [21]:
patient_missing_data

{1: ['Grade', 'Complication_not_leak', 'readmission_30d', 'los'],
 2: ['Grade', 'Complication_not_leak', 'readmission_30d', 'los'],
 3: ['path_distalmargin',
  'Grade',
  'Complication_not_leak',
  'readmission_30d',
  'los'],
 4: ['Grade', 'Complication_not_leak', 'readmission_30d', 'los'],
 5: ['Grade', 'Complication_not_leak', 'readmission_30d', 'los'],
 6: ['Grade', 'Complication_not_leak', 'readmission_30d', 'los'],
 7: ['Grade', 'Complication_not_leak', 'readmission_30d', 'los'],
 8: ['intraop_complications',
  'Grade',
  'Complication_not_leak',
  'readmission_30d',
  'los'],
 9: ['Grade', 'Complication_not_leak', 'readmission_30d', 'los'],
 10: ['Grade', 'Complication_not_leak', 'readmission_30d', 'los'],
 12: ['Grade', 'Complication_not_leak', 'readmission_30d', 'los'],
 13: ['Grade', 'Complication_not_leak', 'readmission_30d', 'los'],
 15: ['Grade', 'Complication_not_leak', 'readmission_30d', 'los'],
 17: ['Grade', 'Complication_not_leak', 'readmission_30d', 'los'],
 18: ['pa

In [22]:
patient_failed_criteria

{79: {'Grade': 'criterion_violated', 'los': 'criterion_violated'},
 89: {'path_circummargin': 'criterion_violated',
  'path_lymph_total': 'criterion_violated',
  'Grade': 'criterion_violated',
  'los': 'criterion_violated'},
 127: {'path_lymph_total': 'criterion_violated'},
 128: {'path_distalmargin': 'criterion_violated',
  'path_circummargin': 'criterion_violated',
  'Grade': 'criterion_violated',
  'readmission_30d': 'criterion_violated',
  'los': 'criterion_violated'},
 141: {'readmission_30d': 'criterion_violated'},
 190: {'Grade': 'criterion_violated', 'los': 'criterion_violated'},
 228: {'intraop_complications': 'criterion_violated',
  'Grade': 'criterion_violated',
  'los': 'criterion_violated',
  'Complication_not_leak': 'criterion_violated'},
 230: {'Grade': 'criterion_violated'},
 261: {'Grade': 'criterion_violated'},
 269: {'Grade': 'criterion_violated',
  'Complication_not_leak': 'criterion_violated',
  'los': 'criterion_violated'},
 279: {'Grade': 'criterion_violated',
  

In [23]:
# Calculate overall textbook outcome percentage
total_patients_with_complete_data = len(patient_outcomes)
textbook_outcome_count = sum(1 for outcome in patient_outcomes.values() if outcome)
textbook_outcome_percentage = (textbook_outcome_count / total_patients_with_complete_data) * 100 if total_patients_with_complete_data > 0 else 0

print(f"total_patients_with_complete_data: {total_patients_with_complete_data}")
print(f"textbook_outcome_count: {textbook_outcome_count}")
print(f"textbook_outcome_percentage: {textbook_outcome_percentage:.2f}%")


total_patients_with_complete_data: 308
textbook_outcome_count: 71
textbook_outcome_percentage: 23.05%


In [24]:
# Calculate overall textbook outcome percentage
total_patients_with_complete_data = len(patient_outcomes)
textbook_outcome_count = sum(1 for outcome in patient_outcomes.values() if outcome)
textbook_outcome_percentage = (textbook_outcome_count / total_patients_with_complete_data) * 100 if total_patients_with_complete_data > 0 else 0

# Calculate percentages for failed criteria
failed_patient_count = total_patients_with_complete_data - textbook_outcome_count
failed_criteria_percentages = {
    criterion: (count / failed_patient_count) * 100 
    for criterion, count in failed_criteria_counts.items()
} if failed_patient_count > 0 else {criterion: 0 for criterion in failed_criteria_counts}

# Total number of patients
total_patients = len(patient_groups)
missing_data_patients = len(patient_missing_data)

# Generate summary statistics
print(f"Total patients in dataset: {total_patients}")
print(f"Patients excluded due to missing data: {missing_data_patients} ({(missing_data_patients / total_patients) * 100:.2f}%)")
print(f"Patients included in analysis (complete data): {total_patients_with_complete_data} ({(total_patients_with_complete_data / total_patients) * 100:.2f}%)")
print(f"Patients with textbook outcome: {textbook_outcome_count} ({textbook_outcome_percentage:.2f}% of analyzed patients)")
print(f"Patients without textbook outcome: {failed_patient_count} ({100 - textbook_outcome_percentage:.2f}% of analyzed patients)")

print("\nBreakdown of failed criteria (among patients without textbook outcome):")
for criterion, count in failed_criteria_counts.items():
    percentage = failed_criteria_percentages[criterion]
    print(f"  {criterion}: {count} patients ({percentage:.2f}%)")

Total patients in dataset: 1270
Patients excluded due to missing data: 962 (75.75%)
Patients included in analysis (complete data): 308 (24.25%)
Patients with textbook outcome: 71 (23.05% of analyzed patients)
Patients without textbook outcome: 237 (76.95% of analyzed patients)

Breakdown of failed criteria (among patients without textbook outcome):
  path_proximalmargin: 11 patients (4.64%)
  path_distalmargin: 15 patients (6.33%)
  path_circummargin: 81 patients (34.18%)
  path_lymph_total: 77 patients (32.49%)
  intraop_complications: 52 patients (21.94%)
  Grade: 230 patients (97.05%)
  Complication_not_leak: 93 patients (39.24%)
  readmission_30d: 46 patients (19.41%)
  los: 221 patients (93.25%)


In [25]:
# Sort criteria by difficulty (most failed to least failed)
sorted_criteria = sorted(
    failed_criteria_counts.items(),
    key=lambda x: x[1],
    reverse=True
)

print("\nCriteria ranked by difficulty (most failed to least failed):")
for criterion, count in sorted_criteria:
    percentage = failed_criteria_percentages[criterion]
    print(f"  {criterion}: {count} patients ({percentage:.2f}%)")


Criteria ranked by difficulty (most failed to least failed):
  Grade: 230 patients (97.05%)
  los: 221 patients (93.25%)
  Complication_not_leak: 93 patients (39.24%)
  path_circummargin: 81 patients (34.18%)
  path_lymph_total: 77 patients (32.49%)
  intraop_complications: 52 patients (21.94%)
  readmission_30d: 46 patients (19.41%)
  path_distalmargin: 15 patients (6.33%)
  path_proximalmargin: 11 patients (4.64%)


In [26]:
# Additional analysis on missing data
print("\nAnalysis of missing data:")
missing_criteria_counts = {criterion: 0 for criterion in has_data_for_criterion.keys()}
for patient_id, missing_criteria_list in patient_missing_data.items():
    for criterion in missing_criteria_list:
        missing_criteria_counts[criterion] += 1

print(f"Missing data patterns (total {missing_data_patients} patients excluded):")
for criterion, count in missing_criteria_counts.items():
    percentage = (count / missing_data_patients) * 100 if missing_data_patients > 0 else 0
    print(f"  {criterion}: {count} patients ({percentage:.2f}%)")


Analysis of missing data:
Missing data patterns (total 962 patients excluded):
  path_proximalmargin: 159 patients (16.53%)
  path_distalmargin: 162 patients (16.84%)
  path_circummargin: 189 patients (19.65%)
  path_lymph_total: 142 patients (14.76%)
  intraop_complications: 343 patients (35.65%)
  Grade: 840 patients (87.32%)
  Complication_not_leak: 840 patients (87.32%)
  readmission_30d: 840 patients (87.32%)
  los: 605 patients (62.89%)
